# Show, Don't Cite: Colab quickstart

Runs the negation-checked PoC benchmark with (a) the built-in control provers, (b) any OpenAI-compatible API, or (c) a local Hugging Face model on a T4 GPU.

**Safety:** model-generated code is executed. Colab is a throwaway VM, so that is fine here; don't do this on a machine you care about without a container.

In [ ]:
!git clone -q https://github.com/Zubair-DLCS/show-dont-cite.git
%cd show-dont-cite
!pip -q install -r requirements.txt
!python -m pytest -q | tail -2

## 1. Controls: oracle and hollow provers
The oracle should be 100% correct. Both hollow provers should fool the single-PoC baseline on every false claim and be caught by the negation check.

In [ ]:
for p in ['oracle', 'hollow-plain', 'hollow-disguised']:
    !python -m sdc run --prover {p} --out results/{p}.jsonl --no-resume | tail -12

## 2. A real model through an OpenAI-compatible API
Groq, OpenRouter, Together, Gemini's OpenAI-compatible endpoint, etc. Pick a model name from your provider's current model list.

In [ ]:
import os, getpass
os.environ['SDC_API_BASE'] = 'https://api.groq.com/openai/v1'   # change for your provider
os.environ['SDC_MODEL'] = 'PUT-MODEL-NAME-HERE'
os.environ['SDC_API_KEY'] = getpass.getpass('API key: ')

In [ ]:
tag = os.environ['SDC_MODEL'].replace('/', '_')
!python -m sdc run --prover api --out results/{tag}.jsonl
!python -m sdc report results/{tag}.jsonl -o results/{tag}.html

## 3. Or a local model (T4, 4-bit)

In [ ]:
!pip -q install transformers accelerate bitsandbytes
!python -m sdc run --prover hf --model Qwen/Qwen2.5-Coder-7B-Instruct --out results/qwen2.5-coder-7b.jsonl

## 4. Compare and view

In [ ]:
!python -m sdc eval results/*.jsonl

In [ ]:
from IPython.display import HTML, display
import glob
latest = sorted(glob.glob('results/*.html'))[-1]
print(latest)
display(HTML(open(latest).read()))

## 5. Answer-engine mode

In [ ]:
!python -m sdc ask "Does standardising features change OLS predictions, and is Python's heapq a max-heap?" --prover api